In [0]:
%sql

CREATE OR REPLACE VIEW staged_claims AS
    SELECT * FROM dev.claims_project.bronze_claims WHERE ingestion_timestamp >= current_timestamp() - INTERVAL 1 DAY

In [0]:
%sql

MERGE INTO dev.claims_project.silver_claims AS target
    USING (
        SELECT 
            TRIM(ClaimID) AS ClaimID,
            TRIM(PatientID) AS PatientID,
            TRIM(ProviderID) AS ProviderID,
            UPPER(TRIM(DiagnosisCode)) AS DiagnosisCode,
            UPPER(TRIM(ProcedureCode)) AS ProcedureCode,
            UPPER(TRIM(ProviderSpecialty)) AS ProviderSpecialty,
            UPPER(TRIM(ProviderLocation)) AS ProviderLocation,
            UPPER(TRIM(ClaimStatus)) AS ClaimStatus,
            UPPER(TRIM(ClaimType)) AS ClaimType,
            TO_DATE(ClaimDate, 'dd-MM-yyyy') AS ClaimDate,
            ROUND(CAST(ClaimAmount AS DOUBLE), 2) AS ClaimAmount,
            TO_DATE(ClaimDate, 'dd-MM-yyyy') + INTERVAL 5 DAYS AS claim_received_date,
            TO_DATE(ClaimDate, 'dd-MM-yyyy') + INTERVAL 15 DAYS AS paid_date,
            DATE_TRUNC('MONTH', TO_DATE(ClaimDate, 'dd-MM-yyyy')) AS claim_month,
            DATEDIFF(TO_DATE(ClaimDate, 'dd-MM-yyyy') + INTERVAL 5 DAYS, TO_DATE(ClaimDate, 'dd-MM-yyyy')) AS claim_lag_days,
            DATEDIFF(TO_DATE(ClaimDate, 'dd-MM-yyyy') + INTERVAL 15 DAYS, TO_DATE(ClaimDate, 'dd-MM-yyyy')) AS payment_lag_days,
            ingestion_timestamp,
            data_source

        FROM staged_claims
    ) AS SOURCE
    ON TARGET.CLAIMID = SOURCE.CLAIMID

    WHEN MATCHED THEN
    UPDATE SET
        target.PatientID = source.PatientID,
        target.ProviderID = source.ProviderID,
        target.DiagnosisCode = source.DiagnosisCode,
        target.ProcedureCode = source.ProcedureCode,
        target.ProviderSpecialty = source.ProviderSpecialty,
        target.ProviderLocation = source.ProviderLocation,
        target.ClaimStatus = source.ClaimStatus,
        target.ClaimType = source.ClaimType,
        target.ClaimDate = source.ClaimDate,
        target.ClaimAmount = source.ClaimAmount,
        target.claim_received_date = source.claim_received_date,
        target.paid_date = source.paid_date,
        target.claim_month = source.claim_month,
        target.claim_lag_days = source.claim_lag_days,
        target.payment_lag_days = source.payment_lag_days,
        target.ingestion_timestamp = source.ingestion_timestamp,
        target.data_source = source.data_source
    WHEN NOT MATCHED THEN

    INSERT (
        ClaimID,
        PatientID,
        ProviderID,
        DiagnosisCode,
        ProcedureCode,
        ProviderSpecialty,
        ProviderLocation,
        ClaimStatus,
        ClaimType,
        ClaimDate,
        ClaimAmount,
        claim_received_date,
        paid_date,
        claim_month,
        claim_lag_days,
        payment_lag_days,
        ingestion_timestamp,
        data_source
    )
    VALUES (
        source.ClaimID,
        source.PatientID,
        source.ProviderID,
        source.DiagnosisCode,
        source.ProcedureCode,
        source.ProviderSpecialty,
        source.ProviderLocation,
        source.ClaimStatus,
        source.ClaimType,
        source.ClaimDate,
        source.ClaimAmount,
        source.claim_received_date,
        source.paid_date,
        source.claim_month,
        source.claim_lag_days,
        source.payment_lag_days,
        source.ingestion_timestamp,
        source.data_source
    );